# Module 31 — Design Fundamentals

## Exercise 1: Requirements before architecture

A design conversation is won or lost before anything is drawn. This notebook is
about the ten minutes that decide it.

You will be given briefs of the kind you actually receive, one sentence long and
missing everything that matters, and you will turn them into requirements
specific enough that two engineers reading them would build the same thing.

| | |
|---|---|
| Time | About 45 minutes |
| You need | This notebook. No libraries |
| Comes after | Module D10, the bridge from data structures to design |

---

## 1. The two kinds of requirement

**Functional requirements** say what the system does. A user can upload a file.
A search returns matching documents. An order can be cancelled before dispatch.

**Non-functional requirements** say how well. Uploads up to 5GB. Search returns
in under 200ms at p99. Cancellation is consistent within one second across
regions.

Both matter, but not equally, and this is the point most engineers get wrong:

> **The functional requirements tell you what to build. The non-functional
> requirements tell you what to build it out of.**

"A user can upload a file" is satisfied by a Python script writing to disk. Add
"five thousand concurrent uploads of up to 5GB, durable across a datacentre
failure" and every part of the design changes, while the functional requirement
has not moved at all.

When someone hands you a brief, the functional part is usually stated and the
non-functional part is almost never stated. That absence is your first job.

---

## 2. Requirements you can be held to

A requirement that cannot be measured is a wish. Compare these.

| Wish | Requirement |
|---|---|
| The site should be fast | Search returns in under 200ms at p99, measured at the load balancer |
| It must be highly available | 99.95 percent monthly, excluding scheduled maintenance windows announced 7 days ahead |
| It should scale | Handle 10x current traffic without redesign; current peak is 4,000 QPS |
| Data must not be lost | Zero acknowledged writes lost on single-node failure; up to 5 minutes on regional failure |

Each right-hand cell contains a number, a unit, and a condition under which it is
judged. That is the whole test. If you cannot write down how you would fail the
requirement, it is not one.

Notice the last one especially. "Data must not be lost" sounds absolute and is
unbuildable. The requirement version admits that regional failure costs you
something, and names the price. Designs get made honest by that sentence.

---

## 3. The questions that change the design

There are hundreds of possible questions. These eight change the answer most
often, and asking them takes two minutes.

1. **Who uses it, and how many of them?** Ten internal staff and ten million
   consumers are different systems with the same description.
2. **Read heavy or write heavy, and by what ratio?** 100:1 read heavy invites
   caching and replicas. 1:1 does not.
3. **How fresh must reads be?** Seconds of staleness acceptable, or must a user
   see their own write immediately? This one question decides more than any other.
4. **What is the peak, relative to the average?** Ticketing and payroll are the
   same average and completely different systems.
5. **What happens if it is down for an hour?** Annoyance, lost revenue, or
   someone is harmed. This sets the availability target, and the budget.
6. **What must never be lost, and what may be?** Payments and analytics events
   do not deserve the same machinery.
7. **How big is the data, in a year and in five?** Fits in memory, fits on one
   machine, or does not.
8. **What is already there?** A greenfield answer to a brownfield question is
   the most common way to be wrong while being technically correct.

**The discipline:** ask, write the answer down, and if the stakeholder does not
know, write down the assumption you are making instead and mark it as one. An
unmarked assumption is a landmine; a marked one is a design decision.

---

## 4. Scope, and how to defend it

Every brief expands. The move that keeps a design honest is stating what is out
of scope, in writing, at the start.

```
IN SCOPE      posting, reading, and deleting messages; direct search by author
OUT OF SCOPE  editing history, full-text search, media attachments, moderation
DEFERRED      threading, which changes the data model; revisit at 100k users
```

Three categories, not two. **Deferred** is the useful one, because it records
that you considered the thing and names the trigger for reconsidering it. That
is the difference between a decision and an oversight, and it is what you will be
asked about.

---

## 5. A worked example

The brief you are given:

> "We need a URL shortener."

The requirements a competent engineer writes in ten minutes:

**Functional**
- Submit a long URL, receive a short one
- Visiting the short URL redirects to the long one
- Optionally choose a custom alias
- The creator can see a click count

**Non-functional**
- 100 million new URLs per month; 10 billion redirects per month
- Redirect latency under 50ms at p99, measured at the edge
- 99.99 percent availability for redirects; 99.9 for creation
- Redirects may serve a click count up to 60 seconds stale
- URLs never expire unless explicitly deleted
- A created URL must be immediately usable by its creator

**Out of scope:** analytics beyond a count, user accounts, spam detection.
**Deferred:** custom domains; revisit if a paying customer asks.

Look at what has happened. Two lines of that list, the 100:1 read-to-write ratio
and the tolerance for a stale click count, have already determined that this is a
cache-fronted read path with an asynchronous counter. No box has been drawn and
the architecture is most of the way decided.

**That is what requirements are for.** Not documentation. They are the thing
that makes the design obvious.

---

## 6. The failure mode, shown

Here is the same brief, answered by someone who started with the architecture.

> "URL shortener. I will use a relational database with a `urls` table, an
> auto-increment id encoded to base 62, an index on the short code, and a
> Redis cache in front. Behind a load balancer, three app servers."

Everything in that paragraph is defensible. It is also unanswerable, because
there is no stated requirement it can be judged against. Ask "why base 62 rather
than a random code" and the honest answer is "it is what I have seen". Ask "does
this survive a regional outage" and there is no target to compare with.

Worse, the auto-increment id makes every created URL guessable in sequence,
which is fine for a link shortener and catastrophic for one used to share
private documents. **The brief did not say which it was, and the design has
already assumed.**

An architecture with no requirements behind it cannot be wrong, which is exactly
what is wrong with it.

---

# Your turn

Four briefs. Treat each as if someone said it to you and then waited.

### Task 1

The brief: **"Build us something so the team can share files."**

Write the three questions from section 3 whose answers would change your design
the most, and say for each what the two extreme answers would imply.

Do not write any requirements yet. This task is only about the questions.

In [ ]:
# ANSWER 1
question_1 = "___"
if_answer_is_one_extreme_1 = "___"
if_answer_is_the_other_1 = "___"

question_2 = "___"
if_answer_is_one_extreme_2 = "___"
if_answer_is_the_other_2 = "___"

question_3 = "___"
if_answer_is_one_extreme_3 = "___"
if_answer_is_the_other_3 = "___"

### Task 2

The brief: **"A notification service for our app."**

Write the full requirement set, in the shape of the worked example: functional,
non-functional with numbers, out of scope, deferred.

Where you do not know a number, invent a plausible one and mark it clearly as an
assumption. Marked assumptions are professional. Silent ones are not.

In [ ]:
# ANSWER 2
FUNCTIONAL = [
    "___",
]

NON_FUNCTIONAL = [
    "___",   # every line needs a number, a unit, and a condition
]

OUT_OF_SCOPE = ["___"]
DEFERRED = ["___"]        # and the trigger that would revisit each
ASSUMPTIONS = ["___"]     # anything you invented rather than were told

### Task 3

Below are six statements. Predict, in the comment beside each, whether it is a
usable requirement or a wish, **before** you look at your reasoning.

Then rewrite every wish as a requirement.

In [ ]:
# ANSWER 3
# 1. "The API should respond quickly."                      prediction: ___
# 2. "Uptime of 99.9% measured monthly at the edge."         prediction: ___
# 3. "The system must be secure."                            prediction: ___
# 4. "Support 50,000 concurrent websocket connections."      prediction: ___
# 5. "Reports should be reasonably up to date."              prediction: ___
# 6. "No acknowledged write is lost on single-node failure." prediction: ___

rewritten = {
    1: "___",
    3: "___",
    5: "___",
}

### Task 4

Go back to your notification service from task 2.

Change exactly one non-functional requirement to its opposite extreme. Freshness
from seconds to minutes, or availability from 99 to 99.99, or scale by a factor
of a thousand.

Then write what that single change does to the design, and say whether the rest
of your requirement set is still coherent. Requirements interact, and finding
that out on paper is considerably cheaper than finding it out later.

In [ ]:
# ANSWER 4
requirement_i_changed = "___"
from_value = "___"
to_value = "___"

what_changes_in_the_design = "___"
what_else_in_my_requirements_no_longer_makes_sense = "___"

---

## Self-check

In [ ]:
def _answer(marker):
    """Find the most recent cell you ran that contains the given marker."""
    try:
        matches = [c for c in _ih if marker in c and "def _answer" not in c]
    except NameError:
        print("Run this in Jupyter or VS Code so the self-check can see your cells.")
        return ""
    return matches[-1] if matches else ""


def check(passed, message):
    print(("PASS  " if passed else "FAIL  ") + message)
    return bool(passed)

NEXT = "Move on to exercise 2, estimation."

a1, a2, a3, a4 = (_answer("# ANSWER 1"), _answer("# ANSWER 2"),
                  _answer("# ANSWER 3"), _answer("# ANSWER 4"))

digits = sum(ch.isdigit() for ch in a2)

results = [
    check(a1 and "___" not in a1, "Task 1: three questions with both extremes"),
    check(a1.count("?") >= 3, "Task 1: they are actually questions"),
    check(a2 and "___" not in a2, "Task 2: every section filled"),
    check(digits >= 8, "Task 2: your non-functional requirements contain numbers"),
    check("ASSUMPTIONS" in a2 and len(a2.split("ASSUMPTIONS")[1]) > 25,
          "Task 2: assumptions marked rather than hidden"),
    check("prediction: ___" not in a3, "Task 3: predicted before reasoning"),
    check(a3 and "___" not in a3, "Task 3: wishes rewritten as requirements"),
    check(a4 and "___" not in a4, "Task 4: the knock-on effect is written down"),
]

print()
failed = results.count(False)
print("%d of %d checks failing. Keep going." % (failed, len(results)) if failed
      else "All %d checks passing. %s" % (len(results), NEXT))

---

## What you learned

- Functional requirements say what to build; non-functional requirements decide
  what to build it out of.
- A requirement has a number, a unit, and a condition under which it is judged.
  If you cannot describe failing it, it is a wish.
- Eight questions change the design more than any others, and asking them costs
  two minutes.
- Unknowns become marked assumptions, never silence.
- Scope has three categories, and **deferred** is the one that distinguishes a
  decision from an oversight.
- Good requirements make the architecture nearly obvious. That is their purpose.
- An architecture with no requirements behind it cannot be judged, which is the
  problem with it.

## Before you move on

- [ ] You wrote questions before writing any requirement.
- [ ] Every non-functional requirement you wrote has a number and a unit.
- [ ] You marked at least one assumption as an assumption.
- [ ] You found at least one place where changing one requirement broke another.

**Next:** exercise 2, where those numbers turn into capacity, and you measure
the latency figures rather than reciting them.